# LIMINA -- 04. Evaluasi Model

Precision@20/Recall@90 hari/AUC lintas enam potret, gerbang keputusan
(regresi logistik vs. rule-based), selisih waktu deteksi sungguhan.
Menulis `artifacts/backtest.json` dan `artifacts/ringkasan_evaluasi.md`.

Seluruhnya membandingkan terhadap Kandidat 1 -- kalau notebook 03 belum
melatihnya (jalur anomali/rule_based_penuh, lihat
`artifacts/keputusan.json`), notebook ini mencetak alasannya dan
melewati evaluasi (tidak menulis backtest.json siklus itu), sama seperti
notebook 03 melewati jalur yang tidak berlaku -- bukan galat. Notebook 05
tetap jalan dengan kandidat yang tersedia.

Akurasi TIDAK dihitung di sini -- lihat `docs/rancangan/metodologi.md` bagian 10.


In [1]:
import sys
from pathlib import Path


def _cari_root(mulai: Path) -> Path:
    for kandidat in [mulai, *mulai.parents]:
        if (kandidat / "limina" / "__init__.py").exists():
            return kandidat
    raise RuntimeError(
        "Tidak menemukan folder 'limina/' di direktori ini atau induknya. "
        "Jalankan notebook dari dalam folder proyek LIMINA."
    )


ROOT = _cari_root(Path.cwd())
sys.path.insert(0, str(ROOT))

import json

import pandas as pd
import joblib

from limina import artifacts_io, config, contracts, metrics, model_registry, models, raw_ingest, report, snapshot, splits

with open(config.PATH_JENDELA) as f:
    jendela = json.load(f)
tanggal_potret = jendela["tanggal_potret"]

panel = pd.read_csv(config.PATH_PANEL, parse_dates=["as_of_date", "event_date", "feature_max_source_date"])
snapshot_dict = {
    tanggal: pd.read_csv(
        config.path_snapshot(tanggal), parse_dates=["as_of_date", "event_date", "feature_max_source_date"]
    )
    for tanggal in tanggal_potret
}

# Prasyarat seluruh notebook ini. Kalau belum ada (jalur anomali/rule_based
# di notebook 03), tiap cell berikut memeriksa ini dan melewati dirinya
# sendiri dengan pesan jelas, bukan FileNotFoundError.
MODEL_1_TERSEDIA = model_registry.model_tersedia()

if MODEL_1_TERSEDIA:
    model_lr, scaler, median_latih = model_registry.muat_model_terlatih()
    try:
        model_gb = joblib.load(config.ARTIFACTS_DIR / "model_kandidat_2.joblib")
    except FileNotFoundError:
        model_gb = None
    with open(config.ARTIFACTS_DIR / "hasil_kebocoran.json") as f:
        hasil_kebocoran = json.load(f)
else:
    model_lr = scaler = median_latih = model_gb = hasil_kebocoran = None
    alasan = ""
    if config.PATH_KEPUTUSAN.exists():
        with open(config.PATH_KEPUTUSAN) as f:
            alasan = f" (keputusan notebook 03: {json.load(f)['keputusan']})"
    print(f"Kandidat 1 belum tersedia{alasan}. Evaluasi dilewati sampai notebook 03 melatihnya.")

print(f"Dievaluasi terhadap {len(tanggal_potret)} potret: {tanggal_potret}")


Kandidat 1 belum tersedia (keputusan notebook 03: anomali_tanpa_label). Evaluasi dilewati sampai notebook 03 melatihnya.
Dievaluasi terhadap 6 potret: ['2026-03-26', '2026-04-25', '2026-05-25', '2026-06-24', '2026-07-24', '2026-08-23']


## 1. Precision@20, Recall@90 hari, AUC -- lintas potret, keempat kandidat

In [2]:
if MODEL_1_TERSEDIA:
    hasil_lintas_potret = snapshot.evaluasi_lintas_potret(
        snapshot_dict, model_lr, model_gb, scaler, median_latih, k=config.K_TOP
    )
    display(
        hasil_lintas_potret.pivot(index="tanggal_potret", columns="kandidat", values="precision_at_20").round(3)
    )
else:
    hasil_lintas_potret = None
    print("Dilewati -- Kandidat 1 belum tersedia.")


Dilewati -- Kandidat 1 belum tersedia.


## 2. Gerbang keputusan

In [3]:
if MODEL_1_TERSEDIA:
    keputusan_gerbang = snapshot.gerbang_keputusan(hasil_lintas_potret)
    print(f"Keputusan : {keputusan_gerbang['keputusan']}")
    print(f"Alasan    : {keputusan_gerbang['alasan']}")
else:
    keputusan_gerbang = None
    print("Dilewati -- Kandidat 1 belum tersedia.")


Dilewati -- Kandidat 1 belum tersedia.


## 3. Selisih waktu deteksi sungguhan

Untuk tiap emiten yang jadi sampel positif di data latih, merekonstruksi skor
mingguan dari data mentah sungguhan sepanjang periode sebelum peristiwanya,
lalu mencari kapan skor itu pertama melewati ambang top-K dan bertahan
(`docs/rancangan/AMBANG-peran-model-dan-evaluasi.md` bagian 4.4).

In [4]:
if MODEL_1_TERSEDIA:
    df_qf = pd.read_csv(config.RAW_DIR / f"{config.TABEL_QUARTERLY_FINANCIALS}.csv")
    df_dt = pd.read_csv(config.RAW_DIR / f"{config.TABEL_DAILY_TRANSACTION}.csv")
    df_dfu = pd.read_csv(config.RAW_DIR / f"{config.TABEL_DAILY_FULL_UNIVERSE_CLOSE}.csv")
    df_harga = raw_ingest.gabungkan_harga(df_dt, df_dfu)
    symbols_universe = sorted(set(df_qf["symbol"]) | set(df_harga["symbol"]))

    positif_latih = panel[panel["is_event_90d"] == 1].drop_duplicates(subset=["symbol", "event_date"])

    hasil_selisih = []
    for _, baris in positif_latih.iterrows():
        tanggal_list = pd.date_range(
            end=pd.Timestamp(baris["event_date"]) - pd.Timedelta(days=1),
            periods=config.RIWAYAT_SKOR_HARI_SEBELUM_EVENT // config.RIWAYAT_SKOR_FREKUENSI_HARI,
            freq=f"{config.RIWAYAT_SKOR_FREKUENSI_HARI}D",
        )
        riwayat_fitur = raw_ingest.bangun_riwayat_fitur_historis(baris["symbol"], tanggal_list, df_qf, df_harga)
        riwayat_fitur = riwayat_fitur.dropna(subset=contracts.KOLOM_FITUR, how="all").reset_index(drop=True)

        if len(riwayat_fitur) < 2:
            hasil_selisih.append({"terdeteksi": False, "tanggal_terdeteksi": None, "selisih_hari": None})
            continue

        riwayat_fitur["skor"] = models.skor_kandidat_1(model_lr, scaler, riwayat_fitur, median_latih).values
        ambang = metrics.kalibrasi_ambang(
            riwayat_fitur["skor"].values, jumlah_top=config.K_TOP,
            total_cakupan=max(len(symbols_universe), config.K_TOP + 1),
        )
        hasil_selisih.append(metrics.hitung_selisih_waktu(riwayat_fitur, ambang, baris["event_date"]))

    ringkasan_selisih = metrics.ringkas_selisih_waktu(hasil_selisih)
    ringkasan_selisih["jumlah_sampel_positif_latih"] = int(len(positif_latih))
    print(ringkasan_selisih)
else:
    ringkasan_selisih = None
    print("Dilewati -- Kandidat 1 belum tersedia.")


Dilewati -- Kandidat 1 belum tersedia.


## 4. Alarm palsu dan kejadian terlewat (untuk backtest.json)

In [5]:
if MODEL_1_TERSEDIA:
    alarm_palsu = []
    kejadian_terlewat_list = []
    for tanggal, snap in snapshot_dict.items():
        skor_lr = models.skor_kandidat_1(model_lr, scaler, snap, median_latih)
        persentil = skor_lr.rank(pct=True) * 100
        ambang_persentil = 100 * (1 - config.K_TOP / len(snap)) if len(snap) else 0
        top_k = snap.loc[persentil >= ambang_persentil]

        alarm_palsu += [
            {"symbol": s, "tanggal_potret": tanggal}
            for s in top_k.loc[top_k["is_event_90d"] == 0, "symbol"].tolist()
        ]
        terlewat = snap[(snap["is_event_90d"] == 1) & (~snap["symbol"].isin(top_k["symbol"]))]
        kejadian_terlewat_list += [
            {"symbol": s, "tanggal_potret": tanggal, "event_date": str(e)}
            for s, e in terlewat[["symbol", "event_date"]].itertuples(index=False)
        ]

    print(f"Alarm palsu (lintas {len(snapshot_dict)} potret): {len(alarm_palsu)}")
    print(f"Kejadian terlewat (lintas {len(snapshot_dict)} potret): {len(kejadian_terlewat_list)}")
else:
    alarm_palsu, kejadian_terlewat_list = [], []
    print("Dilewati -- Kandidat 1 belum tersedia.")


Dilewati -- Kandidat 1 belum tersedia.


## 5. Tulis backtest.json dan ringkasan markdown

In [6]:
if MODEL_1_TERSEDIA:
    backtest_json = artifacts_io.bangun_backtest_json(
        hasil_lintas_potret, ringkasan_selisih, [], alarm_palsu, kejadian_terlewat_list
    )
    artifacts_io.tulis_json(backtest_json, config.ARTIFACTS_DIR / "backtest.json")

    report.tulis_ringkasan_markdown(
        config.ARTIFACTS_DIR / "ringkasan_evaluasi.md",
        hasil_lintas_potret=hasil_lintas_potret,
        ringkasan_selisih=ringkasan_selisih,
        keputusan_gerbang=keputusan_gerbang,
        hasil_kebocoran=hasil_kebocoran,
    )

    print(json.dumps(backtest_json["ringkasan"], indent=2, default=str))
    print(f"\nDitulis: {config.ARTIFACTS_DIR / 'backtest.json'}")
    print(f"Ditulis: {config.ARTIFACTS_DIR / 'ringkasan_evaluasi.md'}")
else:
    print(
        "backtest.json dan ringkasan_evaluasi.md TIDAK ditulis siklus ini -- keduanya "
        "mengevaluasi Kandidat 1 terhadap rule-based. artifacts/scores.json (notebook 05) "
        "tetap diperbarui dengan kandidat yang tersedia."
    )

print("\nLanjut ke notebook 05 (penilaian_dan_artefak) untuk skor pasar hari ini.")


backtest.json dan ringkasan_evaluasi.md TIDAK ditulis siklus ini -- keduanya mengevaluasi Kandidat 1 terhadap rule-based. artifacts/scores.json (notebook 05) tetap diperbarui dengan kandidat yang tersedia.

Lanjut ke notebook 05 (penilaian_dan_artefak) untuk skor pasar hari ini.
